# ChessMind Arena — Chess Policy Model Training

This notebook trains the first chess policy network for ChessMind Arena.

The model learns:

    Chess Position → Strong Candidate Move

The training data comes from the prepared Hugging Face dataset:

    chessmind-arena-policy-dataset

Each example contains:

- FEN
- Stockfish principal-variation first move
- Stockfish evaluation
- search depth
- node count

The first model is a CNN-based policy network.

The model is not intended to replace the LLM.

Instead, it will eventually be used as:

    Chess Position
          ↓
    Chess Policy Network
          ↓
    Top-K Candidate Moves
          ↓
    LLM Player
          ↓
    Selected Move

This notebook establishes the first supervised-learning baseline.

In [1]:
!pip install -q datasets python-chess torch pandas numpy tqdm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 74.2 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


In [2]:
import chess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [28]:
SEED = 42
BATCH_SIZE = 256
learning_rate = 1e-3
EPOCHS = 5

NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

In [4]:
import getpass
import os

os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face Token: ")

In [5]:
HF_DATASET_ID = "Mridul-Dixit/chessmind-arena-policy-dataset"

dataset = load_dataset(HF_DATASET_ID)

print(dataset)

README.md:   0%|          | 0.00/663 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.04MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  417kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/80000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 80000
    })
    validation: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 10000
    })
})


In [6]:
sample = dataset["train"][0]

In [7]:
print("FEN:", sample["fen"])
print("Target move:", sample["target_move"])
print("Depth:", sample["depth"])
print("CP:", sample["cp"])
print("Mate:", sample["mate"])

FEN: 8/1R6/3p1p2/4bP1p/4k3/1R4P1/7r/3K4 w - -
Target move: b7b4
Depth: 65
CP: 0.0
Mate: None


In [8]:
board = chess.Board(sample["fen"])

print(board)

. . . . . . . .
. R . . . . . .
. . . p . p . .
. . . . b P . p
. . . . k . . .
. R . . . . P .
. . . . . . . r
. . . K . . . .


In [9]:
print("\nTarget move:", sample["target_move"])
move = chess.Move.from_uci(sample["target_move"])
print("Is legal:", move in board.legal_moves)


Target move: b7b4
Is legal: True


In [10]:
PIECE_TO_CHANNEL = {
    (chess.WHITE, chess.PAWN): 0,
    (chess.WHITE, chess.KNIGHT): 1,
    (chess.WHITE, chess.BISHOP): 2,
    (chess.WHITE, chess.ROOK): 3,
    (chess.WHITE, chess.QUEEN): 4,
    (chess.WHITE, chess.KING): 5,

    (chess.BLACK, chess.PAWN): 6,
    (chess.BLACK, chess.KNIGHT): 7,
    (chess.BLACK, chess.BISHOP): 8,
    (chess.BLACK, chess.ROOK): 9,
    (chess.BLACK, chess.QUEEN): 10,
    (chess.BLACK, chess.KING): 11,
}


def encode_board(fen):
    board = chess.Board(fen)

    encoded = np.zeros(
        (12, 8, 8),
        dtype=np.float32
    )

    for square, piece in board.piece_map().items():

        channel = PIECE_TO_CHANNEL[
            (piece.color, piece.piece_type)
        ]

        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)

        encoded[channel, row, col] = 1.0

    return encoded

In [11]:
encoded_board = encode_board(sample["fen"])

print("Encoded board shape:", encoded_board.shape)
print("Non zero entries:", np.count_nonzero(encoded_board))

Encoded board shape: (12, 8, 8)
Non zero entries: 11


In [12]:
NUM_SQUARES = 64
BASE_MOVE_CLASSES = 64 * 64

PROMOTION_PIECES = [
    chess.KNIGHT,
    chess.BISHOP,
    chess.ROOK,
    chess.QUEEN,
]

PROMOTION_OFFSET = BASE_MOVE_CLASSES

PROMOTION_TO_INDEX = {
    chess.KNIGHT: 0,
    chess.BISHOP: 1,
    chess.ROOK: 2,
    chess.QUEEN: 3,
}

NUM_MOVE_CLASSES = BASE_MOVE_CLASSES + (
    64 * 64 * len(PROMOTION_PIECES)
)


def encode_move(uci_move):
    move = chess.Move.from_uci(uci_move)

    from_square = move.from_square
    to_square = move.to_square

    base_index = from_square * 64 + to_square

    if move.promotion is None:
        return base_index

    promotion_index = PROMOTION_TO_INDEX[move.promotion]

    promotion_index = (
        promotion_index * 64 * 64
        + base_index
    )

    return PROMOTION_OFFSET + promotion_index


def decode_move(move_index):
    if move_index < BASE_MOVE_CLASSES:

        from_square = move_index // 64
        to_square = move_index % 64

        return chess.Move(
            from_square,
            to_square
        )

    promotion_index = move_index - PROMOTION_OFFSET

    promotion_type = promotion_index // (64 * 64)

    base_index = promotion_index % (64 * 64)

    from_square = base_index // 64
    to_square = base_index % 64

    promotion_piece = PROMOTION_PIECES[promotion_type]

    return chess.Move(
        from_square,
        to_square,
        promotion=promotion_piece
    )

In [13]:
test_moves = [
    "e2e4",
    "g1f3",
    "e7e8q",
    "a7a8n",
]

for move_uci in test_moves:

    index = encode_move(move_uci)
    decoded = decode_move(index)

    print(
        move_uci,
        "→",
        index,
        "→",
        decoded.uci()
    )

e2e4 → 796 → e2e4
g1f3 → 405 → g1f3
e7e8q → 19772 → e7e8q
a7a8n → 7224 → a7a8n


In [14]:
class ChessPolicyDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index):
        row = self.dataset[index]

        board_tensor = encode_board(
            row["fen"]
        )

        target = encode_move(
            row["target_move"]
        )

        return (
            torch.tensor(
                board_tensor,
                dtype=torch.float32
            ),
            torch.tensor(
                target,
                dtype=torch.long
            )
        )

In [15]:
train_dataset = ChessPolicyDataset(
    dataset["train"]
)
val_dataset = ChessPolicyDataset(
    dataset["validation"]
)
test_dataset = ChessPolicyDataset(
    dataset["test"]
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 80000
Validation dataset size: 10000
Test dataset size: 10000


In [16]:
X, y = train_dataset[0]

print("Input shape:", X.shape)
print("Target:", y.item())
print("Target move:", dataset["train"][0]["target_move"])


Input shape: torch.Size([12, 8, 8])
Target: 3161
Target move: b7b4


In [34]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 313
Validation batches: 40
Test batches: 40


In [44]:
class ChessPolicyNet(nn.Module):
    def __init__(self, input_channels=12, num_classes=NUM_MOVE_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.policy_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.policy_head(x)
        return x

In [45]:
model = ChessPolicyNet().to(DEVICE)

print(model)

ChessPolicyNet(
  (features): Sequential(
    (0): Conv2d(12, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
  )
  (policy_head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=20480, bias=True)
  )
)


In [30]:
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

Total parameters: 12906112


In [46]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)   

In [38]:
def calculate_top_k_accuracy(
    logits,
    targets,
    k
):
    _, predictions = torch.topk(
        logits,
        k=k,
        dim=1
    )

    correct = (
        predictions == targets.unsqueeze(1)
    ).any(dim=1)

    return correct.float().mean().item()

In [39]:
def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    total_samples = 0

    top1 = 0
    top3 = 0
    top5 = 0
    top10 = 0

    with torch.no_grad():

        for inputs, targets in loader:

            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)

            logits = model(inputs)

            loss = criterion(
                logits,
                targets
            )

            batch_size = inputs.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size

            top1 += (
                calculate_top_k_accuracy(
                    logits,
                    targets,
                    1
                ) * batch_size
            )

            top3 += (
                calculate_top_k_accuracy(
                    logits,
                    targets,
                    3
                ) * batch_size
            )

            top5 += (
                calculate_top_k_accuracy(
                    logits,
                    targets,
                    5
                ) * batch_size
            )

            top10 += (
                calculate_top_k_accuracy(
                    logits,
                    targets,
                    10
                ) * batch_size
            )

    return {
        "loss": total_loss / total_samples,
        "top1": top1 / total_samples,
        "top3": top3 / total_samples,
        "top5": top5 / total_samples,
        "top10": top10 / total_samples,
    }

In [48]:
history = []
for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    total_samples = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )

    for inputs, targets in progress:
        inputs = inputs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size
        progress.set_postfix({
            "loss": running_loss / total_samples
        })
    train_loss = running_loss / total_samples
    val_metrics = evaluate_model(model, val_loader)
    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        **{
            f"val_{k}": v
            for k, v in val_metrics.items()
        }
    }

    history.append(epoch_result)

    print(f"\nEpoch {epoch + 1}")

    print(f"Train Loss: {train_loss:.4f}")

    print( f"Val Loss: {val_metrics['loss']:.4f}")

    print(f"Top-1: {val_metrics['top1']:.4f}")

    print(f"Top-3: {val_metrics['top3']:.4f}")

    print(f"Top-5: {val_metrics['top5']:.4f}")

    print(f"Top-10: {val_metrics['top10']:.4f}")

Epoch 1/5:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 1
Train Loss: 5.0224
Val Loss: 5.4899
Top-1: 0.0404
Top-3: 0.1056
Top-5: 0.1520
Top-10: 0.2504


Epoch 2/5:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 2
Train Loss: 4.5876
Val Loss: 5.4063
Top-1: 0.0507
Top-3: 0.1138
Top-5: 0.1665
Top-10: 0.2649


Epoch 3/5:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 3
Train Loss: 4.2283
Val Loss: 5.4587
Top-1: 0.0527
Top-3: 0.1224
Top-5: 0.1761
Top-10: 0.2696


Epoch 4/5:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 4
Train Loss: 3.8977
Val Loss: 5.4905
Top-1: 0.0474
Top-3: 0.1147
Top-5: 0.1700
Top-10: 0.2706


Epoch 5/5:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 5
Train Loss: 3.5779
Val Loss: 5.7461
Top-1: 0.0489
Top-3: 0.1191
Top-5: 0.1734
Top-10: 0.2773


In [49]:
test_metrics = evaluate_model(
    model,
    test_loader
)

print("Test Results")
print("------------")

for metric, value in test_metrics.items():

    print(f"{metric}: {value:.4f}")

Test Results
------------
loss: 5.7213
top1: 0.0492
top3: 0.1236
top5: 0.1791
top10: 0.2839


In [26]:
def get_candidate_moves(
    model,
    fen,
    top_k=5
):

    model.eval()

    board = chess.Board(fen)

    encoded = encode_board(fen)

    tensor = torch.tensor(
        encoded,
        dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        logits = model(tensor)[0]

    legal_moves = list(
        board.legal_moves
    )

    legal_indices = []

    for move in legal_moves:

        move_index = encode_move(
            move.uci()
        )

        legal_indices.append(
            (move_index, move)
        )

    scored_moves = []

    for move_index, move in legal_indices:

        score = logits[move_index].item()

        scored_moves.append(
            (move.uci(), score)
        )

    scored_moves.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return scored_moves[:top_k]

In [51]:
test_fen = dataset["test"][0]["fen"]

print(
    chess.Board(test_fen)
)

print(
    "\nStockfish target:",
    dataset["test"][0]["target_move"]
)

candidates = get_candidate_moves(
    model,
    test_fen,
    top_k=5
)

print("\nPolicy candidates:")

for move, score in candidates:

    print(
        f"{move:6s} {score:.4f}"
    )

. r q . . r k .
. . . . . p p p
p . . . . b . .
. . p . . P . .
P . . . Q . . .
. . . . . . . .
. P P . . B P P
R . . . . R K .

Stockfish target: a1b1

Policy candidates:
a1d1   1.7504
g2g3   1.4534
f1d1   1.2113
h2h4   1.1599
e4d3   0.9829
